## Example 1: asyncmy functionality test 1

In [ ]:
import asyncmy

# Example 1: Basic asyncmy MySQL driver functionality
print("Example 1: Basic asyncmy MySQL driver functionality")

try:
    print("asyncmy module imported successfully")
    print("Async MySQL driver for Python")
    
    # Check available components
    attrs = [attr for attr in dir(asyncmy) if not attr.startswith('_')]
    print(f"Available components: {attrs}")
    
    # Show connection-related functionality
    connection_attrs = [attr for attr in attrs if 'connect' in attr.lower()]
    print(f"Connection-related: {connection_attrs}")
    
    # Show cursor/pool related functionality  
    data_attrs = [attr for attr in attrs if any(word in attr.lower() for word in ['cursor', 'pool', 'execute'])]
    print(f"Data operation related: {data_attrs}")
    
    # Check for exception classes
    exception_attrs = [attr for attr in attrs if any(word in attr for word in ['Error', 'Exception'])]
    print(f"Exception classes: {exception_attrs}")
    
    # Test basic connection configuration (without actual connection)
    if hasattr(asyncmy, 'connect'):
        print("✓ connect function available for MySQL connections")
    
    if hasattr(asyncmy, 'Connection'):
        print("✓ Connection class available")
    
    # Show version if available
    if hasattr(asyncmy, '__version__'):
        print(f"asyncmy version: {asyncmy.__version__}")
    
    print("asyncmy example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncmy example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: asyncmy functionality test 2

In [ ]:
import asyncmy

# Example 2: Connection configuration and basic asyncmy usage
print("Example 2: Connection configuration and basic asyncmy usage")

try:
    # Test asyncmy connection configuration (without requiring actual MySQL server)
    async def test_connection_parameters():
        # Show connection parameter structure that asyncmy accepts
        connection_params = {
            "host": "localhost",
            "port": 3306,
            "user": "testuser",
            "password": "testpass",
            "database": "testdb",
            "charset": "utf8mb4",
            "autocommit": False,
            "connect_timeout": 10,
            "read_timeout": None,
            "write_timeout": None,
            "sql_mode": None
        }
        
        print("✓ Connection parameters structure:")
        for key, value in connection_params.items():
            print(f"  {key}: {value}")
        
        # Test connection function signature
        try:
            # This would normally connect to MySQL, but we'll catch the connection error
            # to demonstrate the asyncmy.connect function exists and accepts parameters
            await asyncmy.connect(
                host="nonexistent_host",
                user="test",
                password="test",
                connect_timeout=1  # Short timeout to fail quickly
            )
        except OSError as e:
            # Expected to fail since we're not connecting to a real MySQL server
            print(f"✓ asyncmy.connect() function tested (expected connection failure: {type(e).__name__})")
        except asyncmy.errors.OperationalError as e:
            # asyncmy reports an unresolvable or unreachable host as MySQL client error 2003
            assert e.args[0] == 2003, f"expected MySQL client error 2003, got {e.args!r}"
            print(f"✓ asyncmy.connect() function tested (expected connection failure: {type(e).__name__} {e.args[0]})")
        else:
            raise AssertionError("asyncmy.connect() to 'nonexistent_host' unexpectedly succeeded")
        
        return True
    
    # Test cursor types and options
    async def test_cursor_types():
        print("\n✓ asyncmy cursor types available:")
        
        # Import cursor classes to verify they exist
        from asyncmy.cursors import Cursor, DictCursor, SSCursor, SSDictCursor

        cursor_types = {
            "Cursor": "Standard tuple-based cursor",
            "DictCursor": "Dictionary-based result cursor", 
            "SSCursor": "Server-side cursor for large results",
            "SSDictCursor": "Server-side dictionary cursor"
        }

        for cursor_name, description in cursor_types.items():
            print(f"  {cursor_name}: {description}")

        print("✓ All cursor types imported successfully")

        return True
    
    # Test asyncmy connection context manager pattern
    async def test_connection_patterns():
        print("\n✓ asyncmy connection patterns:")
        
        patterns = [
            "await asyncmy.connect(...) - Direct connection",
            "async with conn.cursor() as cursor - Context manager cursor",
            "await conn.ensure_closed() - Proper connection cleanup",
            "conn.autocommit(True/False) - Transaction control",
            "await cursor.execute(sql, params) - Parameterized queries",
            "await cursor.fetchone/fetchmany/fetchall() - Result fetching"
        ]
        
        for pattern in patterns:
            print(f"  • {pattern}")
        
        return True
    
    # Run all tests
    await test_connection_parameters()
    await test_cursor_types()
    await test_connection_patterns()
    
    print("\nasyncmy connection configuration demonstrated")
    print("asyncmy example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncmy example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: asyncmy functionality test 3

In [ ]:
import asyncmy

# Example 3: asyncmy connection pooling and advanced features
print("Example 3: asyncmy connection pooling and advanced features")

try:
    # Test asyncmy pool creation and configuration
    async def test_asyncmy_pool_creation():
        print("✓ Testing asyncmy connection pool:")
        
        # Test asyncmy.create_pool function exists and accepts parameters
        pool_params = {
            "host": "localhost",
            "port": 3306,
            "user": "testuser", 
            "password": "testpass",
            "database": "testdb",
            "minsize": 5,
            "maxsize": 20,
            "pool_recycle": 3600,
            "charset": "utf8mb4"
        }
        
        print("  Pool configuration parameters:")
        for key, value in pool_params.items():
            print(f"    {key}: {value}")
        
        # Test pool creation against a port guaranteed closed (bind an ephemeral port, read it back,
        # close it): minsize=1 opens a connection up front, so it must fail with OperationalError 2003
        import socket
        with socket.socket() as probe:
            probe.bind(("127.0.0.1", 0))
            closed_port = probe.getsockname()[1]
        try:
            pool = await asyncmy.create_pool(
                host="127.0.0.1",
                port=closed_port,  # nothing listens here
                user="testuser",
                password="testpass",
                minsize=1,
                maxsize=5,
                connect_timeout=1
            )
        except (OSError, ConnectionRefusedError) as e:
            print(f"  ✓ asyncmy.create_pool() function exists and handles errors: {type(e).__name__}")
        except asyncmy.errors.OperationalError as e:
            assert e.args[0] == 2003, f"expected MySQL client error 2003, got {e.args!r}"
            print(f"  ✓ Pool creation function tested: {type(e).__name__} {e.args[0]}")
        else:
            # If successful, clean up -- and fail: nothing listens on that port
            pool.close()
            await pool.wait_closed()
            raise AssertionError(f"asyncmy.create_pool() on closed port {closed_port} unexpectedly succeeded")
        
        return True
    
    # Test asyncmy pool usage patterns
    async def test_pool_usage_patterns():
        print("\n✓ asyncmy pool usage patterns:")
        
        pool_patterns = [
            "pool = await asyncmy.create_pool(**config)",
            "async with pool.acquire() as conn:",
            "    async with conn.cursor() as cursor:",
            "        await cursor.execute(sql)",
            "        results = await cursor.fetchall()",
            "pool.close()",
            "await pool.wait_closed()"
        ]
        
        print("  Connection pool lifecycle:")
        for i, pattern in enumerate(pool_patterns, 1):
            print(f"    {i}. {pattern}")
        
        print("\n  Pool configuration options:")
        pool_options = [
            "minsize: Minimum connections in pool",
            "maxsize: Maximum connections in pool", 
            "pool_recycle: Connection lifetime in seconds",
            "echo: Enable SQL logging",
            "charset: Database character set",
            "autocommit: Default autocommit mode"
        ]
        
        for option in pool_options:
            print(f"    • {option}")
        
        return True
    
    # Test asyncmy replication features
    async def test_replication_features():
        print("\n✓ asyncmy replication features:")
        
        # Test that replication module exists
        from asyncmy.replication import BinLogStream
        print("  ✓ BinLogStream class available for MySQL replication")

        # Show replication usage pattern
        replication_pattern = [
            "from asyncmy.replication import BinLogStream",
            "conn = await asyncmy.connect(host='master')",
            "ctl_conn = await asyncmy.connect(host='master')",
            "stream = BinLogStream(conn, ctl_conn, server_id=1)",
            "async for event in stream:",
            "    process_binlog_event(event)"
        ]

        print("  Replication usage pattern:")
        for step in replication_pattern:
            print(f"    {step}")

        return True
    
    # Test asyncmy error handling and types
    async def test_asyncmy_errors():
        print("\n✓ asyncmy error handling:")
        
        # Import error types from asyncmy
        from asyncmy import errors
        print("  ✓ asyncmy.errors module available")

        # Show available error types
        error_attrs = [attr for attr in dir(errors) if not attr.startswith('_') and 'Error' in attr]
        if error_attrs:
            print("  Available error types:")
            for error_type in error_attrs[:5]:  # Show first 5
                print(f"    • {error_type}")

        # Test connection timeout behavior: an unreachable or timed-out host is OperationalError 2003
        try:
            await asyncmy.connect(
                host="192.0.2.1",  # RFC 5737 test address (non-routable)
                user="test",
                connect_timeout=1
            )
        except asyncmy.errors.OperationalError as e:
            assert e.args[0] == 2003, f"expected MySQL client error 2003, got {e.args!r}"
            print(f"  ✓ Connection timeout handling: {type(e).__name__} {e.args[0]}")
        else:
            raise AssertionError("asyncmy.connect() to the non-routable 192.0.2.1 unexpectedly succeeded")
        
        return True
    
    # Test asyncmy constants and protocol support
    async def test_asyncmy_constants():
        print("\n✓ asyncmy constants and protocol:")
        
        # Test constants module
        from asyncmy import constants
        print("  ✓ asyncmy.constants module available")

        # Show some constant categories
        constant_attrs = [attr for attr in dir(constants) if not attr.startswith('_')]
        if constant_attrs:
            print(f"  Available constants: {len(constant_attrs)} items")
            print(f"  Sample constants: {constant_attrs[:3]}")

        # Show protocol features
        print("\n  Protocol features:")
        protocol_features = [
            "MySQL wire protocol implementation",
            "SSL/TLS encryption support", 
            "Authentication plugin support",
            "Prepared statement caching",
            "Binary result set format",
            "Server-side cursor support",
            "Transaction isolation levels",
            "Character set negotiation"
        ]
        
        for feature in protocol_features:
            print(f"    • {feature}")
        
        return True
    
    # Run all advanced feature tests
    await test_asyncmy_pool_creation()
    await test_pool_usage_patterns()
    await test_replication_features()
    await test_asyncmy_errors()
    await test_asyncmy_constants()
    
    print("\nasyncmy connection pooling and advanced features demonstrated")
    print("asyncmy example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncmy example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: asyncmy functionality test 4

In [ ]:
import asyncmy

# Example 4: asyncmy package capabilities and MySQL async patterns
print("Example 4: asyncmy package capabilities and MySQL async patterns")

try:
    print("asyncmy provides async MySQL database connectivity")
    
    # Package analysis
    attrs = [attr for attr in dir(asyncmy) if not attr.startswith('_')]
    print(f"Available components: {len(attrs)}")
    print(f"Key components: {attrs[:8] if len(attrs) > 8 else attrs}")
    
    # Show version information
    if hasattr(asyncmy, '__version__'):
        print(f"asyncmy version: {asyncmy.__version__}")
    
    # Demonstrate database patterns and best practices
    print("\nDatabase patterns demonstrated:")
    
    # Connection string patterns
    connection_patterns = [
        "Basic connection with host, user, password, database",
        "Connection pooling for high-concurrency applications",
        "Transaction management with begin/commit/rollback",
        "Parameterized queries for SQL injection prevention",
        "Cursor-based result fetching (fetchone, fetchmany, fetchall)",
        "Connection timeout and retry handling",
        "SSL/TLS encryption for secure connections",
        "Character set and collation configuration"
    ]
    
    for pattern in connection_patterns:
        print(f"• {pattern}")
    
    # Show key capabilities
    print("\nKey asyncmy capabilities:")
    capabilities = [
        "🚀 High-performance async MySQL driver",
        "🔗 Connection pooling for scalable applications",
        "💾 Full MySQL protocol support",
        "🛡️ Parameterized queries and prepared statements",
        "📊 Support for all MySQL data types",
        "🔄 Transaction management and autocommit control",
        "🔐 SSL/TLS encryption support",
        "⚡ Optimized for asyncio event loop"
    ]
    
    for capability in capabilities:
        print(f"  {capability}")
    
    # Usage patterns
    print(f"\nCommon usage patterns:")
    patterns = [
        "Web application database layer (FastAPI, aiohttp)",
        "Microservices with shared database pools",
        "Real-time data processing and ETL pipelines",
        "High-frequency trading and financial applications",
        "IoT data ingestion and analytics",
        "Chat applications and message queues",
        "Content management systems",
        "E-commerce transaction processing"
    ]
    
    for pattern in patterns:
        print(f"• {pattern}")
    
    # Performance considerations
    print(f"\nPerformance considerations:")
    considerations = [
        "Use connection pooling to reduce connection overhead",
        "Optimize pool size based on concurrent load",
        "Use prepared statements for repeated queries",
        "Implement proper transaction boundaries",
        "Monitor connection pool utilization",
        "Handle connection timeouts gracefully",
        "Use appropriate fetch methods for result size",
        "Configure MySQL server for async workloads"
    ]
    
    for consideration in considerations:
        print(f"• {consideration}")
    
    # Integration examples
    print(f"\nFramework integration:")
    integrations = [
        "FastAPI with dependency injection for database connections",
        "aiohttp with middleware for connection management",
        "Sanic web framework database integration",
        "Celery async task queue with MySQL backend",
        "Django async views (Django 3.1+)",
        "SQLAlchemy async ORM integration"
    ]
    
    for integration in integrations:
        print(f"• {integration}")
    
    print(f"\nasyncmy enables high-performance async MySQL applications")
    print("asyncmy example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"asyncmy example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")